# Ensembl ↔ UniProt ↔ HGNC crosswalk

Build `reference/gene_protein_map.parquet`, the table that lets the identifier
namespaces of the OmicsFM datasets talk to each other:

| namespace | used by |
|---|---|
| UniProt accession | `proteomics_uniprot`, `bulk_transcriptomics_uniprot`, `sc_transcriptomics_uniprot` |
| Ensembl gene ID | `sc_transcriptomics_ensembl` |
| HGNC symbol | `bulk_transcriptomics_hgnc`, and the scGPT vocabulary |

Any analysis that crosses modalities needs it. Exp1, for instance, selects its
highly-variable universe in protein space and then has to materialise the same
cells in gene space, and has to translate Ensembl IDs into symbols before scGPT
will tokenise them.

**Sources**
- [mygene.info](https://mygene.info) — primary Ensembl → UniProt mapping, which
  separates Swiss-Prot (reviewed) from TrEMBL (unreviewed).
- [UniProt](https://rest.uniprot.org) — bulk human cross-references, used as a
  fallback for genes mygene leaves unmapped.

**Choices for this build**
- One row per Ensembl gene, carrying a single chosen accession. Where a gene maps
  to several, preference is: reviewed *and* in the proteomics model → in the model
  → reviewed → first available. `in_model` records which of those applied, and
  proteins outside the model have no ESM-C embedding.
- Version suffixes are stripped (`ENSG00000139618.17` → `ENSG00000139618`).

Like the ground-truth matrices beside it, this file is **pinned**: mygene.info and
UniProt both change, so a rebuild will not reproduce the published table, and
published results are tied to the shipped one. The write at the end refuses to
overwrite unless you say so explicitly.

## 0. Locate the inputs

Everything resolves from the installed `omicsfm`, so this does not depend on the
working directory. The gene universe and the protein universe both come from the
published datasets — run `omicsfm download` first if `data/` is empty.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from omicsfm.attention import GT_DIR_DEFAULT

GT_DIR = Path(GT_DIR_DEFAULT)          # reference/
ROOT = GT_DIR.parent                   # repository root
OUT = GT_DIR / 'gene_protein_map.parquet'

# Any split will do: the splits of a dataset share one var space, so the 34 MB
# test file yields the same identifiers as the 652 MB train file.
def first_split(dataset, order=('test', 'valid', 'train')):
    for split in order:
        path = ROOT / 'data' / dataset / f'{split}.h5ad'
        if path.exists():
            return path
    raise FileNotFoundError(
        f'no split of {dataset} under {ROOT / "data"}; run: omicsfm download test')


print('reference/ :', GT_DIR)
print('output     :', OUT, '(exists)' if OUT.exists() else '(new)')
PROTEOMICS = first_split('proteomics_uniprot')
SC_ENSEMBL = first_split('sc_transcriptomics_ensembl')
print('proteins   :', PROTEOMICS)
print('genes      :', SC_ENSEMBL)

## 1. The proteomics model's protein universe

Read from `var_names` only — the matrix is never loaded. These are the accessions
that have an ESM-C embedding, and the reason a mapping is preferred when it lands
inside this set.

In [ ]:
import anndata as ad

adata = ad.read_h5ad(PROTEOMICS, backed='r')
model_proteins = {str(v) for v in adata.var_names}
adata.file.close()

print(f'model proteome: {len(model_proteins):,} UniProt accessions')
print('example:', sorted(model_proteins)[:5])

## 2. The Ensembl gene universe

Taken from the single-cell dataset's own `var_names`, which is the set of genes
any downstream analysis can actually encounter.

The originally published table was built from the full CELLxGENE Census gene list
instead. That needs `cellxgene-census` and `tiledbsoma`, which have no Windows
build, and the Census `stable` pointer moves — so the dataset's own gene space is
both reproducible and sufficient. The cell below uses Census when it is available
and falls back otherwise.

In [ ]:
gene_symbols = {}
try:
    import cellxgene_census

    with cellxgene_census.open_soma(census_version='stable') as census:
        var_df = (census['census_data']['homo_sapiens'].ms['RNA']
                  .var.read(column_names=['feature_id', 'feature_name'])
                  .concat().to_pandas())
    ensg_raw = var_df['feature_id'].tolist()
    gene_symbols = dict(zip(var_df['feature_id'], var_df['feature_name']))
    source = 'CELLxGENE Census (stable)'
except ImportError:
    adata = ad.read_h5ad(SC_ENSEMBL, backed='r')
    ensg_raw = [str(v) for v in adata.var_names]
    adata.file.close()
    source = SC_ENSEMBL.name

ensg_ids = sorted({eid.split('.')[0] for eid in ensg_raw})
print(f'gene universe: {len(ensg_ids):,} Ensembl gene IDs, from {source}')
print('example:', ensg_ids[:5])

## 3. Map Ensembl → UniProt with mygene.info

Swiss-Prot and TrEMBL hits are both collected, tagged by `reviewed`, so the
selection step below can prefer reviewed accessions.

In [ ]:
import mygene

mg = mygene.MyGeneInfo()
results = mg.querymany(ensg_ids, scopes='ensembl.gene', fields='uniprot,symbol',
                       species='human', returnall=True, verbose=False)

mapping = {}
for hit in results['out']:
    up = hit.get('uniprot')
    if not up:
        continue
    symbol = hit.get('symbol', '')
    for key, reviewed in (('Swiss-Prot', True), ('TrEMBL', False)):
        value = up.get(key) if isinstance(up, dict) else None
        if not value:
            continue
        for acc in ([value] if isinstance(value, str) else value):
            mapping.setdefault(hit['query'], []).append(
                {'accession': acc, 'reviewed': reviewed, 'symbol': symbol})

print(f'mygene.info mapped {len(mapping):,} / {len(ensg_ids):,} genes')

## 4. Fall back to the UniProt bulk cross-references

One streamed TSV of every human entry with its Ensembl cross-references, cached
under `_downloads/` like the other notebooks' sources.

In [ ]:
import csv
import re
from collections import defaultdict
from io import StringIO

from _common import fetch

unmapped = [eid for eid in ensg_ids if eid not in mapping]
print(f'{len(unmapped):,} genes still unmapped')

if unmapped:
    url = ('https://rest.uniprot.org/uniprotkb/stream'
           '?query=(organism_id:9606)&format=tsv'
           '&fields=accession,xref_ensembl,reviewed,gene_names')
    path = fetch(url, 'uniprot_human_ensembl_xrefs.tsv')

    ensg_to_up = defaultdict(list)
    for row in csv.DictReader(StringIO(path.read_text(encoding='utf-8')), delimiter='\t'):
        names = row.get('Gene Names') or ''
        entry = {'accession': row['Entry'],
                 'reviewed': row['Reviewed'] == 'reviewed',
                 'symbol': names.split()[0] if names else ''}
        for match in re.finditer(r'ENSG\d+', row.get('Ensembl', '') or ''):
            ensg_to_up[match.group()].append(entry)

    recovered = {eid: ensg_to_up[eid] for eid in unmapped if eid in ensg_to_up}
    mapping.update(recovered)
    print(f'UniProt recovered {len(recovered):,} more; total {len(mapping):,}')

## 5. Choose one accession per gene

In order: reviewed and in the model, in the model, reviewed, first available.

In [ ]:
def select(entries):
    for test in (lambda e: e['reviewed'] and e['accession'] in model_proteins,
                 lambda e: e['accession'] in model_proteins,
                 lambda e: e['reviewed']):
        for entry in entries:
            if test(entry):
                return entry
    return entries[0] if entries else None


rows = []
for ensg in ensg_ids:
    best = select(mapping.get(ensg, []))
    if best is None:
        continue
    rows.append({
        'ensembl_gene_id': ensg,
        'uniprot_accession': best['accession'],
        'gene_symbol': best['symbol'] or gene_symbols.get(ensg, ''),
        'reviewed': best['reviewed'],
        'in_model': best['accession'] in model_proteins,
    })

result = pd.DataFrame(rows)
print(f'{len(result):,} genes mapped')
print(f"  reviewed : {int(result['reviewed'].sum()):,}")
print(f"  in model : {int(result['in_model'].sum()):,}")
print(f"  with symbol: {int((result['gene_symbol'] != '').sum()):,}")
result.head()

## 6. Save

Refuses to replace an existing file. The published results are tied to the shipped
table, so re-pinning it means regenerating everything derived from it.

In [ ]:
OVERWRITE = False   # True re-pins the published crosswalk

if OUT.exists() and not OVERWRITE:
    raise FileExistsError(
        f'{OUT} already exists. Published results are pinned to it; set '
        'OVERWRITE = True only if you intend to replace it and regenerate '
        'every result derived from it.')

assert result['ensembl_gene_id'].is_unique, 'one row per gene expected'
result.to_parquet(OUT, index=False)
print(f'saved {OUT.name} ({OUT.stat().st_size / 1e6:.2f} MB, {len(result):,} rows)')
print(f'columns: {list(result.columns)}')